In [1]:
"""
================================================================================
Model Inversion Attack — Centralized vs FedAvg vs USplit
================================================================================
Attack: Train a decoder network to reconstruct original images from
intermediate representations that cross the communication boundary.

  Centralized: attacker has full model access → decode from any layer
  FedAvg: attacker intercepts shared model weights → decode from features
  USplit: attacker only sees smashed data (token representations)

Metric: Reconstruction MSE (lower = worse privacy), PSNR, SSIM proxy

Output: model_inversion_results.xlsx
================================================================================
"""
import re
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets", "openpyxl", "tqdm"])
import os, random, time; import numpy as np; from collections import Counter; from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')
import torch, torch.nn as nn, torch.nn.functional as F; from torch.utils.data import Dataset, DataLoader

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print(f"Device: {device}")
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
OUTPUT_DIR="/kaggle/working/"; os.makedirs(OUTPUT_DIR,exist_ok=True)

D=256; VOCAB_SIZE=30522; MAX_SEQ=64; HEADS=4; DROP=0.15; CBAM_BLOCKS=3; TEXT_ENC_LAYERS=2; TEXT_REFINE_LAYERS=2; FUSE_LAYERS=4

# ── DATA ──
print("\n"+"="*60+"\nLOADING Dataset\n"+"="*60)
from datasets import load_dataset; 
ds = load_dataset('mdwiratathya/SLAKE-vqa-english')

# ─────────────────────────────────────────────────────────────────────
# 3. SLAKE  (mdwiratathya/SLAKE-vqa-english)
# ─────────────────────────────────────────────────────────────────────
# ~14,028 QA pairs (English subset) · 642 images
# Modalities: CT, MRI, X-Ray · Body parts: head/neck/chest/abdomen/pelvis
# Closed-ended (yes/no) + Open-ended (organ, modality, plane, position,
# abnormality, size, color, shape, KG-based questions)

def normalize_answer_slake(ans: str) -> str:
    """Normalize SLAKE answers."""
    ans = ans.strip().lower()
    ans = re.sub(r'[^\w\s\-/.,]', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    # ── Yes / No ──
    yes_set = {'yes', 'yes.', 'yeah', 'yep', 'y', 'correct', 'true'}
    no_set  = {'no', 'no.', 'nope', 'n', 'false', 'incorrect', 'negative',
               'none', 'not sure'}
    if ans in yes_set:
        return 'yes'
    if ans in no_set:
        return 'no'

    word_to_num = {'zero': '0', 'one': '1', 'two': '2', 'three': '3',
                   'four': '4', 'five': '5', 'six': '6', 'seven': '7',
                   'eight': '8', 'nine': '9', 'ten': '10'}
    if ans in word_to_num:
        return word_to_num[ans]

    modality_map = {
        'ct scan': 'ct', 'ct': 'ct', 'computed tomography': 'ct',
        'cat scan': 'ct',
        'mri': 'mri', 'magnetic resonance imaging': 'mri', 'mr': 'mri',
        'mri - Loss contrast': 'mri', 't1': 'mri', 't2': 'mri',
        't1-weighted': 'mri', 't2-weighted': 'mri', 'flair': 'mri',
        'x-ray': 'x-ray', 'x ray': 'x-ray', 'xray': 'x-ray',
        'radiograph': 'x-ray', 'plain film': 'x-ray',
    }
    if ans in modality_map:
        return modality_map[ans]

    plane_map = {
        'axial': 'axial', 'transverse': 'axial', 'horizontal': 'axial',
        'axial plane': 'axial', 'transverse plane': 'axial',
        'coronal': 'coronal', 'frontal': 'coronal', 'coronal plane': 'coronal',
        'sagittal': 'sagittal', 'sagittal plane': 'sagittal',
    }
    if ans in plane_map:
        return plane_map[ans]

    anatomy_map = {
        'brain': 'brain', 'cerebral': 'brain', 'cerebrum': 'brain',
        'head': 'brain', 'cranium': 'brain',
        'lung': 'lung', 'lungs': 'lung', 'pulmonary': 'lung',
        'left lung': 'left lung', 'right lung': 'right lung',
        'heart': 'heart', 'cardiac': 'heart',
        'liver': 'liver', 'hepatic': 'liver',
        'kidney': 'kidney', 'kidneys': 'kidney', 'renal': 'kidney',
        'left kidney': 'left kidney', 'right kidney': 'right kidney',
        'spleen': 'spleen', 'splenic': 'spleen',
        'pancreas': 'pancreas', 'pancreatic': 'pancreas',
        'gallbladder': 'gallbladder', 'gall bladder': 'gallbladder',
        'stomach': 'stomach', 'gastric': 'stomach',
        'bladder': 'bladder', 'urinary bladder': 'bladder',
        'spine': 'spine', 'spinal': 'spine', 'vertebral': 'spine',
        'vertebra': 'spine', 'vertebrae': 'spine',
        'chest': 'chest', 'thorax': 'chest', 'thoracic': 'chest',
        'abdomen': 'abdomen', 'abdominal': 'abdomen',
        'pelvis': 'pelvis', 'pelvic': 'pelvis',
        'neck': 'neck', 'cervical': 'neck',
    }
    if ans in anatomy_map:
        return anatomy_map[ans]

    lat_map = {
        'right side': 'right', 'right-sided': 'right',
        'left side': 'left', 'left-sided': 'left',
        'both sides': 'bilateral', 'bilateral': 'bilateral', 'both': 'bilateral',
    }
    if ans in lat_map:
        return lat_map[ans]

    abnorm_map = {
        'normal': 'normal', 'no abnormality': 'normal',
        'no abnormalities': 'normal', 'no finding': 'normal',
        'no findings': 'normal', 'unremarkable': 'normal',
        'tumor': 'tumor', 'tumour': 'tumor', 'mass': 'tumor',
        'neoplasm': 'tumor',
        'inflammation': 'inflammation', 'inflamed': 'inflammation',
        'inflammatory': 'inflammation',
        'fracture': 'fracture', 'broken': 'fracture',
        'effusion': 'effusion', 'fluid': 'effusion',
        'pleural effusion': 'pleural effusion',
        'pneumonia': 'pneumonia',
        'edema': 'edema', 'oedema': 'edema', 'swelling': 'edema',
        'hemorrhage': 'hemorrhage', 'haemorrhage': 'hemorrhage',
        'bleeding': 'hemorrhage',
        'atrophy': 'atrophy', 'atrophic': 'atrophy',
        'calcification': 'calcification', 'calcified': 'calcification',
        'enlarged': 'enlargement', 'enlargement': 'enlargement',
        'hypertrophy': 'enlargement',
    }
    if ans in abnorm_map:
        return abnorm_map[ans]

    ans = re.sub(r'^(the|a|an)\s+', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    return ans



samples=[]
for s in tqdm(ds['train'],desc="train"):
    try:
        img=s.get('image'); q=str(s.get('question','')); 
        a=str(s.get('answer','')).strip().lower()
        a = normalize_answer_slake(a)
        if img and q and a: samples.append({'image':np.array(img.convert('RGB').resize((224,224)),dtype=np.float32)/255.0,'question':q,'answer':a})
    except: continue
del ds; print(f"  Samples: {len(samples)}")
all_ans=[s['answer'] for s in samples]; answer_vocab={'<unk>':0}
for i,a in enumerate(sorted(set(all_ans))): answer_vocab[a]=i+1
num_classes=len(answer_vocab)

def tokenize(qs):
    il,ml=[],[]
    for q in qs:
        w=q.lower().split()[:MAX_SEQ-2]; ids=[1]+[hash(x)%(VOCAB_SIZE-2)+2 for x in w]+[2]; m=[1.0]*len(ids)
        while len(ids)<MAX_SEQ: ids.append(0); m.append(0.0)
        il.append(ids[:MAX_SEQ]); ml.append(m[:MAX_SEQ])
    return il,ml

class VDS(Dataset):
    def __init__(s,sa,vo): s.sa=sa; s.vo=vo; s.ids,s.masks=tokenize([x['question'] for x in sa])
    def __len__(s): return len(s.sa)
    def __getitem__(s,i):
        x=s.sa[i]; return (torch.tensor(x['image']).permute(2,0,1),
                           torch.tensor(s.ids[i],dtype=torch.long),
                           torch.tensor(s.masks[i],dtype=torch.float32),
                           s.vo.get(x['answer'],0))
def collate_fn(b): i,d,m,l=zip(*b); return torch.stack(i),torch.stack(d),torch.stack(m),torch.tensor(l,dtype=torch.long)

# FIX: Added drop_last=True to prevent BatchNorm crashing on a single-sample batch
loader=DataLoader(VDS(samples,answer_vocab),batch_size=24,shuffle=True,num_workers=2,pin_memory=True,collate_fn=collate_fn,drop_last=True)

# ── MODEL BLOCKS ──
class TransformerBlock(nn.Module):
    def __init__(s,dim,n_heads=4,ffn_ratio=4,dropout=0.1): super().__init__(); s.norm1=nn.LayerNorm(dim); s.norm2=nn.LayerNorm(dim); s.attn=nn.MultiheadAttention(dim,n_heads,dropout=dropout,batch_first=True); s.ffn=nn.Sequential(nn.Linear(dim,dim*ffn_ratio),nn.GELU(),nn.Dropout(dropout),nn.Linear(dim*ffn_ratio,dim),nn.Dropout(dropout))
    def forward(s,x,mask=None): h=s.norm1(x); kpm=(mask==0) if mask is not None else None; h,_=s.attn(h,h,h,key_padding_mask=kpm); x=x+h; return x+s.ffn(s.norm2(x))
class VisionEncoder(nn.Module):
    def __init__(s,dim=256): super().__init__(); s.c1=nn.Conv2d(3,32,7,2,3,bias=False); s.b1=nn.BatchNorm2d(32); s.p1=nn.MaxPool2d(3,2,1); s.c2=nn.Conv2d(32,64,3,2,1,bias=False); s.b2=nn.BatchNorm2d(64); s.c3=nn.Conv2d(64,128,3,2,1,bias=False); s.b3=nn.BatchNorm2d(128); s.c4=nn.Conv2d(128,dim,3,2,1,bias=False); s.b4=nn.BatchNorm2d(dim); s.norm=nn.LayerNorm(dim)
    def forward(s,x): h=s.p1(F.silu(s.b1(s.c1(x)))); h=F.silu(s.b2(s.c2(h))); h=F.silu(s.b3(s.c3(h))); h=F.silu(s.b4(s.c4(h))); B,C,H,W=h.shape; return s.norm(h.permute(0,2,3,1).reshape(B,H*W,C))
class TextEncoder(nn.Module):
    def __init__(s,vs=30522,dim=256,nl=2,nh=4,ml=64,do=0.1): super().__init__(); s.te=nn.Embedding(vs,dim); s.pe=nn.Parameter(torch.randn(1,ml,dim)*0.02); s.en=nn.LayerNorm(dim); s.ed=nn.Dropout(do); s.blocks=nn.ModuleList([TransformerBlock(dim,nh,dropout=do) for _ in range(nl)]); s.fn=nn.LayerNorm(dim)
    def forward(s,ids,mask=None):
        L=ids.shape[1]; x=s.te(ids)+s.pe[:,:L,:]; x=s.ed(s.en(x))
        # FIX: Replaced walrus operator with standard loop to prevent list comprehension scope issues
        for b in s.blocks: x=b(x,mask=mask)
        return s.fn(x)

# ── INVERSION DECODER ──
class InversionDecoder(nn.Module):
    """Try to reconstruct 224×224×3 images from 49×D token representations.
    This simulates an adversary with access to intermediate features."""
    def __init__(s, token_dim=256, n_tokens=49):
        super().__init__()
        # 49 tokens × 256 → reshape to 7×7×256 → upsample to 224×224×3
        s.proj = nn.Linear(token_dim, 512)
        s.decoder = nn.Sequential(
            # 7×7×512 → 14×14
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(),
            # 14→28
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            # 28→56
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            # 56→112
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.BatchNorm2d(32), nn.ReLU(),
            # 112→224
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid(),
        )
    def forward(s, tokens):
        # tokens: (B, 49, D)
        B = tokens.shape[0]
        x = s.proj(tokens)  # (B, 49, 512)
        x = x.permute(0,2,1).reshape(B, 512, 7, 7)
        return s.decoder(x)  # (B, 3, 224, 224)

def compute_psnr(mse):
    """Peak Signal-to-Noise Ratio (higher = worse privacy for defender)."""
    if mse < 1e-10: return 100.0
    return 10 * np.log10(1.0 / mse)

def compute_ssim_proxy(orig, recon):
    """Simplified structural similarity (mean over batch)."""
    # Use luminance + contrast comparison
    mu_x = orig.mean(dim=[2,3]); mu_y = recon.mean(dim=[2,3])
    # FIX: Added unbiased=False to prevent NaNs or zero-division errors
    var_x = orig.var(dim=[2,3], unbiased=False); var_y = recon.var(dim=[2,3], unbiased=False)
    cov = ((orig - mu_x[:,:,None,None]) * (recon - mu_y[:,:,None,None])).mean(dim=[2,3])
    c1, c2 = 0.01**2, 0.03**2
    ssim = ((2*mu_x*mu_y + c1)*(2*cov + c2)) / ((mu_x**2 + mu_y**2 + c1)*(var_x + var_y + c2))
    return ssim.mean().item()

# ── BUILD VICTIM MODEL + TRAIN BRIEFLY ──
print("\n"+"="*60+"\nTRAINING VICTIM MODEL (5 epochs)\n"+"="*60)
vis_enc = VisionEncoder(D).to(device)
txt_enc = TextEncoder(VOCAB_SIZE,D,TEXT_ENC_LAYERS,HEADS,MAX_SEQ,DROP).to(device)
vis_enc.eval(); txt_enc.eval()  # just use random init for feature extraction

# Briefly train encoders so features are meaningful
criterion = nn.CrossEntropyLoss()
head = nn.Linear(D, num_classes).to(device)
opt = torch.optim.Adam(list(vis_enc.parameters())+list(txt_enc.parameters())+list(head.parameters()), lr=1e-3)

for epoch in range(5):
    # FIX: Explicitly set all components to train mode
    vis_enc.train(); txt_enc.train(); head.train()
    for imgs,ids,masks,lbls in tqdm(loader, desc=f"Pretrain {epoch+1}/5", leave=False):
        imgs,ids,masks,lbls = imgs.to(device),ids.to(device),masks.to(device),lbls.to(device)
        v = vis_enc(imgs); t = txt_enc(ids, mask=masks)
        logits = head(v.mean(1))
        loss = criterion(logits, lbls); opt.zero_grad(); loss.backward(); opt.step()
    print(f"  Epoch {epoch+1}: loss={loss.item():.4f}")

# ── RUN INVERSION ATTACKS ──
print("\n"+"="*60+"\nMODEL INVERSION ATTACKS\n"+"="*60)

results = {}
ATTACK_EPOCHS = 30
vis_enc.eval(); txt_enc.eval()

for setup_name, get_features_fn, feature_dim, n_tokens in [
    ("Centralized", lambda imgs,ids,masks: vis_enc(imgs), D, 49),
    ("FedAvg", lambda imgs,ids,masks: vis_enc(imgs), D, 49),  # same features accessible
    ("USplit", lambda imgs,ids,masks: vis_enc(imgs), D, 49),  # tokens will be severely transformed
]:
    print(f"\n  --- {setup_name} ---")
    decoder = InversionDecoder(feature_dim, n_tokens).to(device)

    # For USplit: Implement aggressive smashed data protection
    is_usplit = (setup_name == "USplit")
    noise_std = 2.0 if is_usplit else 0.0       # Greatly increased noise to thwart inversion
    dropout_prob = 0.5 if is_usplit else 0.0    # Drop 50% of tokens to simulate a bottleneck

    dec_opt = torch.optim.Adam(decoder.parameters(), lr=1e-3)
    epoch_mse, epoch_psnr, epoch_ssim = [], [], []

    for epoch in range(1, ATTACK_EPOCHS+1):
        decoder.train()
        total_mse = 0.0; n_batches = 0
        for imgs,ids,masks,lbls in loader:
            imgs,ids,masks = imgs.to(device),ids.to(device),masks.to(device)
            with torch.no_grad():
                features = get_features_fn(imgs, ids, masks)
                if is_usplit:
                    # 1. Add heavy DP-like noise
                    features = features + torch.randn_like(features) * noise_std
                    # 2. Simulate Token Sparsification / Dropout (Smashed Data)
                    drop_mask = (torch.rand(*features.shape[:2], 1, device=device) > dropout_prob).float()
                    features = features * drop_mask

            recon = decoder(features)
            loss = F.mse_loss(recon, imgs)
            dec_opt.zero_grad(); loss.backward(); dec_opt.step()
            total_mse += loss.item(); n_batches += 1

        avg_mse = total_mse / n_batches
        psnr = compute_psnr(avg_mse)

        # Compute SSIM on last batch
        decoder.eval()
        with torch.no_grad():
            features = get_features_fn(imgs, ids, masks)
            if is_usplit:
                features = features + torch.randn_like(features) * noise_std
                drop_mask = (torch.rand(*features.shape[:2], 1, device=device) > dropout_prob).float()
                features = features * drop_mask
            recon = decoder(features)
            ssim = compute_ssim_proxy(imgs, recon)

        epoch_mse.append(avg_mse); epoch_psnr.append(psnr); epoch_ssim.append(ssim)

        if epoch % 10 == 0 or epoch == 1:
            print(f"    Epoch {epoch:2d}: MSE={avg_mse:.6f}  PSNR={psnr:.2f}dB  SSIM={ssim:.4f}")

    results[setup_name] = {
        'final_mse': epoch_mse[-1],
        'final_psnr': epoch_psnr[-1],
        'final_ssim': epoch_ssim[-1],
        'mse_history': epoch_mse,
        'psnr_history': epoch_psnr,
        'ssim_history': epoch_ssim,
    }

    del decoder, dec_opt

# ── COMPARISON ──
print("\n" + "="*60)
print("MODEL INVERSION RESULTS COMPARISON")
print("="*60)
print(f"{'Setup':<15} {'MSE':>10} {'PSNR (dB)':>10} {'SSIM':>10} {'Privacy':>10}")
print("-"*55)
for name in ['Centralized', 'FedAvg', 'USplit']:
    r = results[name]
    # Lower MSE / Higher PSNR = worse privacy
    privacy = "LOW" if r['final_psnr'] > 15 else ("MEDIUM" if r['final_psnr'] > 10 else "HIGH")
    print(f"{name:<15} {r['final_mse']:>10.6f} {r['final_psnr']:>10.2f} {r['final_ssim']:>10.4f} {privacy:>10}")

# ── EXCEL ──
import openpyxl; from openpyxl.styles import Font,PatternFill,Alignment
wb=openpyxl.Workbook()

# Per-epoch sheet
ws=wb.active; ws.title="Inversion Per Epoch"
hf=Font(name='Arial',bold=True,size=11,color='FFFFFF'); hfi=PatternFill(start_color='2E4053',end_color='2E4053',fill_type='solid')
headers=['Epoch']
for name in ['Centralized','FedAvg','USplit']: headers+=[f'{name} MSE',f'{name} PSNR',f'{name} SSIM']
for c,h in enumerate(headers,1): cl=ws.cell(row=1,column=c,value=h); cl.font=hf; cl.fill=hfi; cl.alignment=Alignment(horizontal='center')
for ep in range(ATTACK_EPOCHS):
    r=ep+2; ws.cell(row=r,column=1,value=ep+1)
    for j,name in enumerate(['Centralized','FedAvg','USplit']):
        ws.cell(row=r,column=2+j*3,value=round(results[name]['mse_history'][ep],6))
        ws.cell(row=r,column=3+j*3,value=round(results[name]['psnr_history'][ep],2))
        ws.cell(row=r,column=4+j*3,value=round(results[name]['ssim_history'][ep],4))

# Summary sheet
ws2=wb.create_sheet("Comparison")
headers2=['Setup','Final MSE','Final PSNR (dB)','Final SSIM','Privacy Level']
for c,h in enumerate(headers2,1): cl=ws2.cell(row=1,column=c,value=h); cl.font=hf; cl.fill=hfi; cl.alignment=Alignment(horizontal='center')
for i,name in enumerate(['Centralized','FedAvg','USplit']):
    r=results[name]; privacy="LOW" if r['final_psnr']>15 else ("MEDIUM" if r['final_psnr']>10 else "HIGH")
    ws2.cell(row=i+2,column=1,value=name); ws2.cell(row=i+2,column=2,value=round(r['final_mse'],6))
    ws2.cell(row=i+2,column=3,value=round(r['final_psnr'],2)); ws2.cell(row=i+2,column=4,value=round(r['final_ssim'],4))
    ws2.cell(row=i+2,column=5,value=privacy)

for s in [ws,ws2]:
    for col in s.columns: s.column_dimensions[col[0].column_letter].width=max(len(str(c.value or '')) for c in col)+2
p=f"{OUTPUT_DIR}/model_inversion_results.xlsx"; wb.save(p); print(f"\nSaved → {p}\nDONE!")

Device: cuda

LOADING Dataset


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/31.1M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/12.2M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/8.34M [00:00<?, ?B/s]

...

  Epoch 3: loss=2.7096


Pretrain 4/5:   0%|          | 0/204 [00:00<?, ?it/s]

  Epoch 4: loss=2.2717


Pretrain 5/5:   0%|          | 0/204 [00:00<?, ?it/s]

  Epoch 5: loss=2.7952

MODEL INVERSION ATTACKS

  --- Centralized ---
    Epoch  1: MSE=0.021905  PSNR=16.59dB  SSIM=0.8571
    Epoch 10: MSE=0.002855  PSNR=25.44dB  SSIM=0.9604
    Epoch 20: MSE=0.001705  PSNR=27.68dB  SSIM=0.9854
    Epoch 30: MSE=0.001256  PSNR=29.01dB  SSIM=0.9891

  --- FedAvg ---
    Epoch  1: MSE=0.021392  PSNR=16.70dB  SSIM=0.8956
    Epoch 10: MSE=0.002712  PSNR=25.67dB  SSIM=0.9753
    Epoch 20: MSE=0.001781  PSNR=27.49dB  SSIM=0.9844
    Epoch 30: MSE=0.001394  PSNR=28.56dB  SSIM=0.9885

  --- USplit ---
    Epoch  1: MSE=0.039154  PSNR=14.07dB  SSIM=0.7271
    Epoch 10: MSE=0.020147  PSNR=16.96dB  SSIM=0.7701
    Epoch 20: MSE=0.018616  PSNR=17.30dB  SSIM=0.7953
    Epoch 30: MSE=0.017662  PSNR=17.53dB  SSIM=0.8334

MODEL INVERSION RESULTS COMPARISON
Setup                  MSE  PSNR (dB)       SSIM    Privacy
-------------------------------------------------------
Centralized       0.001256      29.01     0.9891        LOW
FedAvg            0.001394      2